# M2 Baseline — Text-to-SQL on Spider dev (Google Colab T4)

**Goal:** Benchmark 3 candidate models on Spider dev using execution accuracy (EX).

**Models tested:** Qwen2.5-Coder-7B-Instruct · defog/sqlcoder-7b-2 · OmniSQL-7B  
**Split:** Spider dev

**Workflow per model:**
1. Change `MODEL_ID` in section 6
2. Runtime → Restart and run all
3. Results auto-saved to Drive
4. Repeat for next model
5. Run section 11 to compare all 3


## 0. Runtime check — must show Tesla T4

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# If no GPU shown: Runtime > Change runtime type > T4 GPU > Save

## 1. Install dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece python-dotenv sqlglot wandb pandas pyarrow

## 2. Mount Google Drive

Your Spider data must be uploaded to Drive first. Expected layout:
```
MyDrive/text2sql_data/spider_data/dev.json
MyDrive/text2sql_data/spider_data/database/<db_id>/<db_id>.sqlite
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure paths and clone repo

In [ ]:
import os, sys

# ── EDIT THESE ────────────────────────────────────────────────────────────
GITHUB_TOKEN = ''   # GitHub PAT if repo is private; else leave empty
REPO_SLUG    = 'phuocNg964/text2sql-post-training'  # owner/repo
HF_TOKEN     = ''   # HuggingFace token (needed if model is gated)
WANDB_KEY    = ''   # Weights & Biases API key (optional)
# ──────────────────────────────────────────────────────────────────────────

REPO_DIR  = '/content/text2sql-post-training'
DATA_DIR  = '/content/drive/MyDrive/text2sql_data'
PRED_DIR  = '/content/drive/MyDrive/text2sql_predictions'
CACHE_DIR = '/content/drive/MyDrive/hf_cache'  # persist model weights across sessions
for d in [PRED_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(REPO_DIR):
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git'
    else:
        clone_url = f'https://github.com/{REPO_SLUG}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')

sys.path.insert(0, REPO_DIR)
print('Repo ready:', REPO_DIR)

## 4. Authenticate (HuggingFace + W&B)

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('HF authenticated')

import wandb
if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
    print('W&B authenticated')
else:
    print('W&B skipped (no key)')

## 5. Sanity check — gold SQL must give EX = 1.0

Run this once before any model. If it fails, there is a data path bug.

In [ ]:
from src.data.loader import load_spider
from src.eval.evaluator import evaluate

_records = load_spider(DATA_DIR, split='dev', n=20)
_gold    = [r['gold_sql'] for r in _records]
_result  = evaluate(_records, _gold)

print(f"Gold-SQL sanity check -> EX = {_result['execution_accuracy']}  (expected: 1.0)")
assert _result['execution_accuracy'] == 1.0, 'Data pipeline bug — fix before benchmarking!'
print('Sanity check passed')

## 6. Select model

Change `MODEL_ID`, then **Runtime > Restart and run all** for the next model.

| # | Model | Notes |
|---|---|---|
| 1 | `Qwen/Qwen2.5-Coder-7B-Instruct` | Chat template, our SFT/RL target |
| 2 | `mistralai/Mistral-7B-Instruct-v0.3` | Chat template, strong general baseline |
| 3 | `Qwen/Qwen3-8B` | Chat template, thinking **off** (`enable_thinking=False`) |
| 4 | `Qwen/Qwen3-8B` | Chat template, thinking **on** (`enable_thinking=True`) |

In [ ]:
# ── CHANGE THIS for each run ──────────────────────────────────────────────
MODEL_ID        = 'Qwen/Qwen2.5-Coder-7B-Instruct'       # run 1
# MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'             # run 2
# MODEL_ID = 'Qwen/Qwen3-8B'                                   # run 3 & 4

ENABLE_THINKING = False  # run 3: False, run 4: True (Qwen3 only)

N_SAMPLES = 50     # 50 ≈ 15-20 min; use 500 for the final baseline run
# ─────────────────────────────────────────────────────────────────────────

RUN_NAME = MODEL_ID.split('/')[-1].lower().replace('-', '_')
thinking_tag = '_thinking' if ENABLE_THINKING else ''
OUT_FILE = os.path.join(PRED_DIR, f'{RUN_NAME}{thinking_tag}_spider_dev.jsonl')
RES_FILE = os.path.join(PRED_DIR, f'{RUN_NAME}{thinking_tag}_spider_dev_results.json')
print(f'Model    : {MODEL_ID}')
print(f'Thinking : {ENABLE_THINKING}')
print(f'Samples  : {N_SAMPLES}')
print(f'Output   : {OUT_FILE}')

## 7. Load model (4-bit NF4)

Expected VRAM after load: ~5–6 GB of 15 GB T4.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    cache_dir=CACHE_DIR,
)
model.eval()

allocated = torch.cuda.memory_allocated() / 1024**3
total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'Model loaded. VRAM: {allocated:.1f} GB / {total_mem:.1f} GB')

## 8. Run inference

In [ ]:
import json, re
from src.data.loader import load_spider
from src.data.formatter import build_prompt_from_record

SYSTEM_PROMPT = (
    'You are a SQL expert. Given a database schema and a natural language question, '
    'write a valid SQL query that answers the question. '
    'Output only the SQL query with no explanation.'
)

def parse_sql(text):
    """Extract SQL from ```sql ... ``` block, or return stripped text."""
    m = re.search(r'```(?:sql)?\s*(.*?)```', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()

records = load_spider(DATA_DIR, split='dev', n=N_SAMPLES)
print(f'Loaded {len(records)} Spider dev records')

predictions = []
for i, record in enumerate(records):
    user_prompt = build_prompt_from_record(record)

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_prompt},
    ]
    tmpl_kwargs = {'tokenize': False, 'add_generation_prompt': True}
    if 'qwen3' in MODEL_ID.lower():
        tmpl_kwargs['enable_thinking'] = ENABLE_THINKING
    text = tokenizer.apply_chat_template(messages, **tmpl_kwargs)

    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    predicted_sql = parse_sql(generated)
    predictions.append({'predicted_sql': predicted_sql})

    if (i + 1) % 10 == 0 or i == 0:
        print(f'  [{i+1:>3}/{len(records)}] {record["db_id"]} | {predicted_sql[:80]}')

with open(OUT_FILE, 'w', encoding='utf-8') as f:
    for p in predictions:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print(f'Saved {len(predictions)} predictions to {OUT_FILE}')

## 9. Evaluate

In [ ]:
import json
from src.eval.evaluator import evaluate
from collections import defaultdict

predicted_sqls = [json.loads(l)['predicted_sql'] for l in open(OUT_FILE)]
result = evaluate(records, predicted_sqls)
exec_errors = sum(1 for r in result['results'] if r['execution_error'])

print('=' * 48)
print(f"  Model    : {MODEL_ID}")
print(f"  Split    : spider_dev")
print(f"  Samples  : {N_SAMPLES}")
print('-' * 48)
print(f"  EX       : {result['execution_accuracy']:.4f}")
print(f"  Correct  : {result['n_correct']} / {result['n_total']}")
print(f"  Exec err : {exec_errors} / {result['n_total']}")
print('=' * 48)

by_diff = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in result['results']:
    d = r['difficulty'] or 'unknown'
    by_diff[d]['total'] += 1
    if r['correct']:
        by_diff[d]['correct'] += 1

print('\nBreakdown by difficulty:')
for d, v in sorted(by_diff.items()):
    ex = v['correct'] / v['total'] if v['total'] else 0
    print(f"  {d:<14}: {ex:.4f}  ({v['correct']}/{v['total']})")

with open(RES_FILE, 'w') as f:
    json.dump({'model': MODEL_ID, 'split': 'spider_dev', 'enable_thinking': ENABLE_THINKING, **result}, f, indent=2)
print(f'\nFull results -> {RES_FILE}')

## 10. Log to Weights & Biases (optional)

In [ ]:
if WANDB_KEY:
    import wandb
    wandb.init(
        project='text2sql-post-training',
        name=f'm2-baseline-{RUN_NAME}-spider-dev',
        config={
            'model': MODEL_ID, 'split': 'spider_dev',
            'enable_thinking': ENABLE_THINKING, 'n_samples': N_SAMPLES,
            'quantization': 'nf4-4bit',
        },
    )
    wandb.log({
        'eval/spider_dev/execution_accuracy': result['execution_accuracy'],
        'eval/spider_dev/n_correct':          result['n_correct'],
        'eval/spider_dev/exec_errors':        exec_errors,
        **{f'eval/spider_dev/{d}_ex': v['correct']/v['total'] for d, v in by_diff.items()},
    })
    wandb.finish()
    print('Logged to W&B.')
else:
    print('W&B skipped (no key set).')

## 11. Compare all 3 models

Run after all model result files are saved to Drive. Picks the baseline.

In [ ]:
import json, os, glob

rows = []
for path in sorted(glob.glob(os.path.join(PRED_DIR, '*_results.json'))):
    with open(path) as f:
        d = json.load(f)
    rows.append({
        'model':   d['model'],
        'EX':      d['execution_accuracy'],
        'correct': d['n_correct'],
        'total':   d['n_total'],
        'errors':  sum(1 for r in d['results'] if r['execution_error']),
    })

rows.sort(key=lambda r: -r['EX'])
print(f"{'Model':<45} {'EX':>6}  {'Correct':>8}  {'Errors':>7}")
print('-' * 75)
for r in rows:
    print(f"{r['model']:<45} {r['EX']:>6.4f}  {r['correct']:>3}/{r['total']:<3}  {r['errors']:>7}")

if rows:
    print(f"\nBaseline candidate: {rows[0]['model']}")